## Create Table to summarize results

This notebook creates a table with the normalized NLLHs for different models. It expects to find pickled predictions for 10 folds with the following filenames:

 * ```<scenario_name>.floc.<dist_name>.<fold>.100.<model>.pkl```

`eval_model.py` automatically creates such files. 

**NOTE:** The first cell contains some variables that need to be adjusted. Evaluating all scenarios might take a while.

In [ ]:
import os
import pickle
import sys

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import scipy.stats as scst
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import tabulate

#% matplotlib inline

sys.path.append("../")
from helper import load_data, preprocess, data_source_release

mean_exp = []  #["rf", "nn"]
sc_dict = data_source_release.get_sc_dict()

# 1) Select the distribution you want to fit
floc_exp = ["lognormal_distfit", "lognormal_distfit.rf2", "lognormal_distfit.rf", "lognormal_nn", "lognormal_xgb"]
distribution_hndl = scst.distributions.lognorm
# floc_exp = ["expon_distfit", "expon_distfit.rf2", "expon_distfit.rf", "expon_nn", "expon_xgb"]
# distribution_hndl = scst.distributions.expon
# floc_exp = ["invgauss_distfit", "invgauss_distfit.rf2", "invgauss_distfit.rf", "invgauss_nn", "invgauss_xgb"]
# distribution_hndl = scst.distributions.invgauss
# floc_exp = ["weibull_distfit", "weibull_distfit.rf2", "weibull_distfit.rf", "weibull_nn", "weibull_xgb"]
# distribution_hndl = scst.distributions.weibull_min

# 2) Change the following to existing paths
save_dir_output = "../results_all"
os.makedirs(save_dir_output, exist_ok=True)

output_dir = "../scripts/TEST_100/"
save_dir = "../results"

output_dir_xgb = "../../../Implementation_XGBoost/Discrete-master/scripts/TEST_100_SMAC_NAS_100/"
save_dir_xgb = "../../../Implementation_XGBoost/Discrete-master/results_SMAC_NAS_100/"


# 3) Select whether to compute results for all scenarios or just a single scenario
SCENARIOS = []
SCENARIOS += ['clasp_factoring', ]
SCENARIOS += ['saps-CVVAR', ]
SCENARIOS += ['lpg-zeno', ]
SCENARIOS += ['yalsat_qcp', ]
SCENARIOS += ['spear_qcp', ]
SCENARIOS += ['spear_swgcp', ]
SCENARIOS += ['yalsat_swgcp', ]

load = True

In [ ]:
res_dict = dict()
list_dirs = [output_dir, output_dir_xgb]
for output_dir in list_dirs:
    for fl in os.listdir(output_dir):
        if not ".100." in fl:
            # Select only one seed
            continue
        print(fl, output_dir)

        fl_path = os.path.join(output_dir, fl)
        
        if os.path.isdir(fl_path):
            continue
        
        if not "pkl" in fl_path:
            continue
        
        if "txt" in fl_path or "png" in fl_path:
            continue

        if not os.path.isfile(fl_path):
            fl_path = os.path.join(output_dir)

        with open(fl_path, "rb") as fh:
            pkl = pickle.load(fh)
            train_pred = np.array(pkl[0])
            val_pred = np.array(pkl[1])
            print(train_pred.shape, val_pred.shape)
            add_info = pkl[2]
            task = add_info["task"]
            model = add_info["model"]
            scenario = add_info["scenario"]
            fold = add_info["fold"]

            if "nn" in add_info["model"] and add_info["task"] == "floc":
                print("%20s" % scenario, model, fold, add_info["loaded"])
            
            if "xgb" in add_info["model"]:
                # Transform the xgboostlss parameter predictions
                if 'expon' in model:
                    train_pred = 1/train_pred
                    val_pred = 1/val_pred
                elif 'lognormal' in model:
                    # Flip the parameter order
                    train_pred = train_pred[:, ::-1]
                    val_pred = val_pred[:, ::-1]
                    # Convert log of location -> location
                    train_pred[:, 1] = np.exp(train_pred[:, 1])
                    val_pred[:, 1] = np.exp(val_pred[:, 1])
                elif 'weibull' in model:
                    # Flip the parameter order
                    train_pred = train_pred[:, ::-1]
                    val_pred = val_pred[:, ::-1]
            
        if scenario not in res_dict:
            res_dict[scenario] = dict()

        if model not in floc_exp and model not in mean_exp:
            continue

        if task not in res_dict[scenario]:
            res_dict[scenario][task] = dict()
            
        if model not in res_dict[scenario][task]:
            res_dict[scenario][task][model] = dict()

        res_dict[scenario][task][model][fold] = (train_pred, val_pred)


if len(SCENARIOS) == 0:
    SCENARIOS = list(res_dict.keys())

In [ ]:
for scen in sorted(SCENARIOS):
    print("### %s" % scen)
    try:
        for model in sorted(res_dict[scen]["mean"]):
            print("mean %20s" % model, len(res_dict[scen]["mean"][model]))
    except:
        pass
    for model in sorted(res_dict[scen]["floc"]):
        print("floc %20s" % model, len(res_dict[scen]["floc"][model]))

In [ ]:
nllh_dict = dict()
mean_dict = dict()
var_dict = dict()
if load:
    with open(save_dir + "/" + distribution_hndl.name + ".nllh.pkl", "rb") as fh:
        nllh_dict2 = pickle.load(fh)
        
    with open(save_dir + "/" + distribution_hndl.name + ".mean.pkl", "rb") as fh:
        mean_dict2 = pickle.load(fh)
        
    with open(save_dir + "/" + distribution_hndl.name + ".var.pkl", "rb") as fh:
        var_dict2 = pickle.load(fh)
    print("LOADED")

    with open(save_dir_xgb + "/" + distribution_hndl.name + ".nllh.pkl", "rb") as fh:
        nllh_dict = pickle.load(fh)
        for scen in nllh_dict.keys():
            if scen not in nllh_dict2.keys():
                continue
            nllh_dict[scen] = nllh_dict[scen] | nllh_dict2[scen]
        
    with open(save_dir_xgb + "/" + distribution_hndl.name + ".mean.pkl", "rb") as fh:
        mean_dict = pickle.load(fh)
        for scen in mean_dict.keys():
            if scen not in mean_dict2.keys():
                continue
            mean_dict[scen] = mean_dict[scen] | mean_dict2[scen]
        
    with open(save_dir_xgb + "/" + distribution_hndl.name + ".var.pkl", "rb") as fh:
        var_dict = pickle.load(fh)
        for scen in var_dict.keys():
            if scen not in var_dict2.keys():
                continue
            var_dict[scen] = var_dict[scen] | var_dict2[scen]
    print("LOADED")
    
    with open(save_dir_output + "/" + distribution_hndl.name + ".nllh.pkl", "wb") as fh:
        pickle.dump(nllh_dict, fh)

    with open(save_dir_output + "/" + distribution_hndl.name + ".mean.pkl", "wb") as fh:
        pickle.dump(mean_dict, fh)

    with open(save_dir_output + "/" + distribution_hndl.name + ".var.pkl", "wb") as fh:
        pickle.dump(var_dict, fh)
    print("dumped to " + save_dir_output)

else:
    print("NOT LOADED")

    

In [ ]:
# Generate NLLH table

# nan means there are some folds missing
# inf means there was no results file found
tab_data = list()
header = ["sc"] + floc_exp
for scen in sorted(nllh_dict.keys()):
    dummy_array = np.ones([2, 2]) * np.inf
    tab_data.append([scen] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) for m in header[1:]])
    tab_data.append(["", ] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) for m in header[1:]])

a = tabulate.tabulate(tab_data, headers=header, floatfmt="5.3f")
print(a)

with open("%s/nllh_%s.txt" % (save_dir_output, distribution_hndl.name), "w") as fh:
    fh.write(a)

## Generate Latex

In [ ]:
tabular_data = list()
scenario_dicts = {
    "clasp_factoring": "Clasp-factoring",
    "saps-CVVAR": "Saps-CV-VAR",
    "lpg-zeno": "LPG-Zenotravel",
    "yalsat_qcp": "YalSAT-QCP",
    "spear_qcp": "Spear-QCP",
    "spear_swgcp": "Spear-SWGCP",
    "yalsat_swgcp": "YalSAT-SWGCP",
}
model_name = {
    "lognormal_distfit": "fitted",
    "lognormal_distfit.rf2": "iRF",
    "lognormal_distfit.rf": "mRF",
    "lognormal_nn": "DistNet",
    "lognormal_xgb": "XGBoostLSS",

    "invgauss_distfit": "fitted",
    "invgauss_distfit.rf2": "iRF",
    "invgauss_distfit.rf": "mRF",
    "invgauss_nn": "DistNet",
    "invgauss_xgb": "XGBoostLSS",

    "weibull_distfit": "fitted",
    "weibull_distfit.rf2": "iRF",
    "weibull_distfit.rf": "mRF",
    "weibull_nn": "DistNet",
    "weibull_xgb": "XGBoostLSS",
}

header = ["Scenario"] + [""] + floc_exp
# header.insert(-1, "Bayes DistNet")

for scen in sorted(nllh_dict.keys()):
    dummy_array = np.ones([2, 2]) * np.inf
    tabular_data.append([scenario_dicts.get(scen, scen)] + ["train"] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) for m in header[2:]])
    tabular_data.append(["", ] + ["test"] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) for m in header[2:]])

latex_header = ["Scenario"] + [""] + [model_name[m] for m in floc_exp]
# latex_header.insert(-1, "Bayes DistNet")

latex_tabular = tabulate.tabulate(tabular_data, tablefmt="latex", floatfmt="5.3f", headers=latex_header)
print(latex_tabular)

with open(f"{save_dir_output}/{floc_exp[0].split('_')[0]}_tabular.tex", "w") as fh:
    fh.write(latex_tabular)

## Generate Boxplots

In [ ]:
# Create dataframes for test data
import pandas as pd
import seaborn as sns
from matplotlib.ticker import LogFormatter, LogLocator, FuncFormatter

scenario_dicts = {
    "clasp_factoring": "Clasp-factoring",
    "saps-CVVAR": "Saps-CV-VAR",
    "lpg-zeno": "LPG-Zenotravel",
    "yalsat_qcp": "YalSAT-QCP",
    "spear_qcp": "Spear-QCP",
    "spear_swgcp": "Spear-SWGCP",
    "yalsat_swgcp": "YalSAT-SWGCP",
}

model_name = {
    "lognormal_distfit": "fitted",
    "lognormal_distfit.rf2": "iRF",
    "lognormal_distfit.rf": "mRF",
    "lognormal_nn": "DistNet",
    "lognormal_xgb": "XGB",

    "invgauss_distfit": "fitted",
    "invgauss_distfit.rf2": "iRF",
    "invgauss_distfit.rf": "mRF",
    "invgauss_nn": "DistNet",
    "invgauss_xgb": "XGB",

    "weibull_distfit": "fitted",
    "weibull_distfit.rf2": "iRF",
    "weibull_distfit.rf": "mRF",
    "weibull_nn": "DistNet",
    "weibull_xgb": "XGB",
}
df = pd.DataFrame(columns=['SC','model', 'NLLH'])
# CAP_NUM = 1e10
for SC in SCENARIOS:
    dummy_array = np.ones([2, 2]) * np.inf
    for model in floc_exp:
        nllh_vals = nllh_dict[SC].get(model, dummy_array)[:, 1]
        # nllh_vals = [nllh if nllh < CAP_NUM else CAP_NUM for nllh in nllh_vals]
        for nllh in nllh_vals:
            df = pd.concat([df, pd.DataFrame({
                'SC': [SC],
                'model': model_name.get(model, model),
                'NLLH': [nllh]})], ignore_index=True)


In [ ]:
# Choose one algorithm to create boxplot for
plt.rcParams.update({
    "font.size": 16,      # default text size
    "axes.titlesize": 18, # subplot titles
    "axes.labelsize": 16, # x/y labels
    "xtick.labelsize": 18,
    "ytick.labelsize": 16,
    "legend.fontsize": 16
})
os.makedirs("./plots/boxplots/", exist_ok=True)
SCENARIOS_ = []
SCENARIOS_ += ['clasp_factoring', ]
SCENARIOS_ += ['saps-CVVAR', ]
SCENARIOS_ += ['lpg-zeno', ]
SCENARIOS_ += ['yalsat_qcp', ]
SCENARIOS_ += ['spear_qcp', ]
SCENARIOS_ += ['spear_swgcp', ]
SCENARIOS_ += ['yalsat_swgcp', ]
for SC in SCENARIOS_:
    # Subset dataframe
    df_algo = df[(df["SC"] == SC) & (~df["model"].isin(["iRF","mRF"]))].copy() #  

    plt.figure(figsize=(8,5))
    ax = sns.boxplot(
        data=df_algo,
        x="model",
        y="NLLH",
    )
    
    num_ticks = 5
    ax.set_yscale("symlog", linthresh=1, linscale=1) # 
    
    ymin, ymax = ax.get_ylim()
    ticks = np.linspace(ymin, ymax, num_ticks)
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"{t:.2f}" for t in ticks])

    # ticks = ax.get_yticks()
    # main_ticks = []
    # for i, t in enumerate(ticks):
    #     if i % 1 == 0: 
    #         main_ticks.append(t)
    # ax.set_yticks(main_ticks)
    # ax.minorticks_off()

    ax.set_ylabel("Test NLLH (log scale)")
    ax.set_xlabel("Model")
    ax.set_title(f"{scenario_dicts.get(SC, SC)} | {floc_exp[0].split('_')[0].upper()}")

    plt.tight_layout()
    plt.savefig(f"./plots/boxplots/{floc_exp[0].split('_')[0]}_{SC}.pdf", dpi=300)